# Public Records — loading the data

Loading the 9 CSV files from `data/public_records/` and a preview of each. Everything is loaded as text (`dtype=str`), with no automatic conversion, to see the raw data.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

DATA_DIR = Path("data/public_records")

tables = {p.stem: pd.read_csv(p, dtype=str) for p in sorted(DATA_DIR.glob("*.csv"))}

list(tables)

['institutional_memos',
 'licensed_facilities',
 'licensed_professionals',
 'officer_filings',
 'public_and_investigative_documents',
 'public_appointments',
 'registered_entities',
 'regulatory_case_events',
 'site_access_case_events']

## Inventory

In [2]:
pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1]}
    for name, df in tables.items()
]).set_index("table")

,rows,columns
table,,
institutional_memos,30,12
licensed_facilities,3,12
licensed_professionals,53,10
officer_filings,136,12
public_and_investigative_documents,40,11
public_appointments,62,12
registered_entities,24,10
regulatory_case_events,233,16
site_access_case_events,413,15


## Preview of each table

### Helper functions

- `clean_table`: steps shared by every table (whitespace, `none` → NA, duplicates, dates). It prints what it fixes.
- `standardize_dates`: tries a fixed list of formats (`DATE_FORMATS`, most frequent first) instead of letting pandas guess. An `assert` checks that no date is lost.
- `split_year_band`: splits a year (`1961`) or a decade (`1920–1929`) into two bounds. The separator is an en dash `–`, not a hyphen `-`.
- `check_foreign_key`: counts references that cannot be found in another table.
- `missing_report`: number and % of missing values, for the columns that have some.

Non-ISO dates found: `1986/01/01`, `2019/03/26 09:00:00`, `08-Jun-2022 10:00:00`, `01/07/1994` and `01/01/1974`. The last two are ambiguous: we read them as day/month (see `licensed_professionals`).

In [3]:
DATE_FORMATS = [
    "%Y-%m-%d",
    "%Y-%m-%dT%H:%M:%S",
    "%Y/%m/%d",
    "%Y/%m/%d %H:%M:%S",
    "%d-%b-%Y %H:%M:%S",
    "%d/%m/%Y",
]

def standardize_dates(series: pd.Series, formats: list[str] = DATE_FORMATS) -> pd.Series:
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in formats:
        todo = parsed.isna() & series.notna()
        parsed[todo] = pd.to_datetime(series[todo], format=fmt, errors="coerce")
    return parsed

def trim_all_text_columns(df: pd.DataFrame, exclude: list[str] = []) -> pd.DataFrame:
    df = df.copy()
    text_cols = df.select_dtypes(include=['object', 'str']).columns.difference(exclude)
    df[text_cols] = df[text_cols].apply(
        lambda col: col.map(lambda x: ' '.join(str(x).split()) if pd.notna(x) else x)
    )
    return df

def convert_to_category(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        df[col] = df[col].astype('category')
    return df

def none_to_na(series: pd.Series) -> pd.Series:
    return series.replace(r'(?i)^none$', pd.NA, regex=True)

def nulls_to_na_all(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_cols = df.select_dtypes(include=['object', 'str']).columns
    df[text_cols] = df[text_cols].apply(none_to_na)
    return df

def drop_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates()

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().sum()
    missing = missing[missing > 0]
    return pd.DataFrame({'missing': missing, '%': (missing / len(df) * 100).round(1)})

def clean_table(df: pd.DataFrame, key: str, date_cols: list[str] = [], trim_exclude: list[str] = [], replace_none: bool = True) -> pd.DataFrame:
    n = len(df)
    trimmed = trim_all_text_columns(df, exclude=trim_exclude)
    changed = (trimmed != df) & df.notna()
    for col in changed.columns[changed.any()]:
        print(f"- Whitespace fixed in {col}: {changed[col].sum()} value(s), {df[col].nunique()} -> {trimmed[col].nunique()} distinct")
    df = trimmed
    if replace_none:
        df = nulls_to_na_all(df)
    df = drop_duplicate_rows(df)
    print(f"- Duplicate rows removed: {n - len(df)}")
    print(f"- {key} unique: {'yes' if df[key].is_unique else 'NO'}")
    for col in date_cols:
        before = df[col].notna().sum()
        df[col] = standardize_dates(df[col])
        assert df[col].notna().sum() == before, col
    if date_cols:
        print(f"- Dates converted: {', '.join(date_cols)}")
    return df

def split_year_band(series: pd.Series) -> pd.DataFrame:
    bounds = series.str.split('–', expand=True)
    low = pd.to_numeric(bounds[0], errors='coerce').astype('Int64')
    high = pd.to_numeric(bounds[1], errors='coerce').astype('Int64').fillna(low)
    return pd.DataFrame({'birth_year_min': low, 'birth_year_max': high, 'birth_year_is_exact': low == high})

def check_foreign_key(series: pd.Series, reference: pd.Series, name: str) -> pd.Series:
    unmatched = series.notna() & ~series.isin(reference)
    filled = series.notna().sum()
    print(f"- {name}: {filled} filled, {unmatched.sum()} not found ({unmatched.sum() / filled:.0%}), {series.isna().sum()} empty")
    return unmatched

### `registered_entities`

In [4]:
reg_ent = tables["registered_entities"].copy()
reg_ent.head()

,registration_number,legal_name,registered_entity_type,registration_date,current_status,registered_address,jurisdiction,declared_activity,parent_registration_number,last_filing_date
0,REG-VLN-01001,Meridian BioLogistics,logistics_company,1996-01-15,active,Meridian Valenne Headquarters,Valenne Republic,International biological and cold-ch...,NaN,2025-03-31
1,REG-VLN-01002,Asterion Clinical Institute,research_institute,1989-01-15,active,Asterion Central Clinical Campus,Valenne Republic,Private preventive-medicine and rese...,NaN,2025-03-31
2,REG-TMB-01003,Northstar Cold Chain Services,logistics_company,1992-01-15,active,Northstar Tembara Cold-Chain Hub,Tembara Commonwealth,Refrigerated transport provider,NaN,2025-03-31
3,REG-VLN-01004,Whitehaven Strategic Holdings,consulting_company,1987-01-15,active,Meridian Valenne Headquarters,Valenne Republic,Investment and consulting company,NaN,2025-03-31
4,REG-VLN-01005,Valenne Industrial Futures Council,trade_association,1984-01-15,active,Valenne Central District,Valenne Republic,Industry-policy association,NaN,2025-03-31


In [5]:
reg_ent = clean_table(reg_ent, 'registration_number', date_cols=['registration_date', 'last_filing_date'])
reg_ent = convert_to_category(reg_ent, ['registered_entity_type', 'current_status', 'jurisdiction'])
display(missing_report(reg_ent))

- Duplicate rows removed: 0
- registration_number unique: yes
- Dates converted: registration_date, last_filing_date


,missing,%
parent_registration_number,24,100.0


**Findings**: 24 entities, no duplicates. `parent_registration_number` is always empty: nothing to impute.

### `licensed_facilities`

In [6]:
lic_fac = tables["licensed_facilities"]
lic_fac.head()

,facility_license_number,facility_name,facility_category,operating_organization_registration,street_address,region,latitude,longitude,license_start,license_end,facility_status,declared_services
0,FAC-VLN-101,Asterion Central Clinical Campus,research_hospital,REG-VLN-01002,"Asterion Central Clinical Campus, Va...",Valenne Republic,47.2,7.51,2001-02-01,NaN,active,preventive medicine; cardiac assessm...
1,FAC-AVL-201,Asterion Avelora Executive Clinic,private_clinic,REG-VLN-01002,"Asterion Avelora Executive Clinic, E...",Avelora Federation,7.35,101.62,2008-05-15,NaN,active,executive health; specialist follow-...
2,FAC-AVL-301,Threshold Earth Avelora Laboratory,research_laboratory,REG-AVL-01007,"Threshold Earth Avelora Laboratory, ...",Avelora Federation,7.5,101.75,2012-01-18,NaN,active,environmental health research; non-i...


In [7]:
lic_fac = clean_table(lic_fac, 'facility_license_number', date_cols=['license_start', 'license_end'])
lic_fac = convert_to_category(lic_fac, ['facility_category', 'region', 'facility_status'])

lic_fac['latitude'] = pd.to_numeric(lic_fac['latitude'], errors='coerce')
lic_fac['longitude'] = pd.to_numeric(lic_fac['longitude'], errors='coerce')
bad_coords = ~lic_fac['latitude'].between(-90, 90) | ~lic_fac['longitude'].between(-180, 180)
print(f"- Invalid coordinates: {bad_coords.sum()}")

check_foreign_key(lic_fac['operating_organization_registration'], reg_ent['registration_number'], 'operating_organization_registration')

lic_fac['declared_services'] = lic_fac['declared_services'].str.split(r'\s*;\s*').apply(tuple)

display(missing_report(lic_fac))
lic_fac

- Duplicate rows removed: 0
- facility_license_number unique: yes
- Dates converted: license_start, license_end
- Invalid coordinates: 0
- operating_organization_registration: 3 filled, 0 not found (0%), 0 empty


,missing,%
license_end,3,100.0


,facility_license_number,facility_name,facility_category,operating_organization_registration,street_address,region,latitude,longitude,license_start,license_end,facility_status,declared_services
0,FAC-VLN-101,Asterion Central Clinical Campus,research_hospital,REG-VLN-01002,"Asterion Central Clinical Campus, Va...",Valenne Republic,47.20,7.51,2001-02-01,NaT,active,"(preventive medicine, cardiac assess..."
1,FAC-AVL-201,Asterion Avelora Executive Clinic,private_clinic,REG-VLN-01002,"Asterion Avelora Executive Clinic, E...",Avelora Federation,7.35,101.62,2008-05-15,NaT,active,"(executive health, specialist follow..."
2,FAC-AVL-301,Threshold Earth Avelora Laboratory,research_laboratory,REG-AVL-01007,"Threshold Earth Avelora Laboratory, ...",Avelora Federation,7.50,101.75,2012-01-18,NaT,active,"(environmental health research, non-..."


**Findings**: 3 facilities, valid coordinates, all linked to a known entity. Empty `license_end` = licence still running.

### `licensed_professionals`

In [8]:
lic_pro = tables["licensed_professionals"]
lic_pro.head()

,license_number,professional_name,profession,specialty,license_issue_date,license_expiry_date,license_status,primary_facility_license,registered_region,disciplinary_status
0,LIC-VLN-MED-1001,Emil Warden,physician,cardiology,1895-07-01,NaN,active,FAC-VLN-101,Valennan,none
1,LIC-VLN-MED-1002,Dr. Naima Orrel,physician,preventive medicine,2002-07-01,NaN,active,FAC-VLN-101,Valennan,none
2,LIC-VLN-MED-1003,Dr. Corin Hale,physician,cardiac imaging,2002-07-01,NaN,active,FAC-AVL-201,Valennan,none
3,LIC-VLN-MED-1004,Dr. Lysa Quen,physician,medical audit,1995-07-01,NaN,active,FAC-VLN-101,Valennan,none
4,LIC-VLN-MED-1005,Faris Uren,physician,cardiology,1986-07-01,NaN,active,FAC-VLN-101,Valennan,none


In [9]:
lic_pro = clean_table(lic_pro, 'license_number', date_cols=['license_issue_date', 'license_expiry_date'], replace_none=False)
lic_pro = convert_to_category(lic_pro, ['profession', 'specialty', 'license_status', 'registered_region', 'disciplinary_status'])

check_foreign_key(lic_pro['primary_facility_license'], lic_fac['facility_license_number'], 'primary_facility_license')

display(missing_report(lic_pro))

- Whitespace fixed in specialty: 1 value(s), 16 -> 15 distinct
- Duplicate rows removed: 0
- license_number unique: yes
- Dates converted: license_issue_date, license_expiry_date
- primary_facility_license: 27 filled, 0 not found (0%), 26 empty


,missing,%
license_expiry_date,53,100.0
primary_facility_license,26,49.1


**Findings**

- Whitespace: one specialty counted twice (`'  environmental  assessment '`), 16 → 15.
- `disciplinary_status`: `none` means "no sanction", not "missing". We keep it (`replace_none=False`).
- `license_expiry_date`: always empty, which fits with all 53 licences being active.
- `01/07/1994`: read as 1 July, like the 52 other issue dates.
- `primary_facility_license`: 26 empty (no main facility), no broken link.
- To look at later: two licences from 1886 and 1895 are still active.

### `officer_filings`

In [10]:
off_fil = tables["officer_filings"]
print(off_fil.shape)
off_fil.head()

(136, 12)


,filing_id,registration_number,officer_local_ref,full_name,birth_year_or_age_band,nationality_or_region,role_title,appointment_date,cessation_date,filing_date,service_address,filing_status
0,FLG-000001,REG-NDH-01015,OFR-20001,Edward Vann,1815,Nordhaven,Industrial and shipping adviser,1840-01-01,1888-12-31,1840-03-09,Nordhaven Union,historical
1,FLG-000002,REG-VLN-01016,OFR-20002,Emil Warden,1871,Valennan,Commercial and medical investor,1895-01-01,1943-12-31,1895-01-13,Valenne Republic,historical
2,FLG-000003,REG-NDH-01017,OFR-20003,Adrian Vosmere,1920–1929,Nordhaven,Biotechnology and logistics executive,1952-01-01,1991-12-31,1952-01-16,Nordhaven Union,historical
3,FLG-000004,REG-VLN-01001,OFR-20004,Dr. Elias Voss,1961,Valennan,Executive chair of Meridian BioLogis...,1996-01-01,NaN,1996-02-15,Meridian Valenne Headquarters,active
4,FLG-000005,REG-KNT-01018,OFR-20005,J. Varlan,1862,Kintaran,Natural-history correspondent,1885-01-01,1918-12-31,1885-03-14,Kintara Republic,historical


In [11]:
off_fil = clean_table(off_fil, 'filing_id', date_cols=['appointment_date', 'cessation_date', 'filing_date'])

off_fil = off_fil.join(split_year_band(off_fil['birth_year_or_age_band']))

off_fil['is_active'] = off_fil['cessation_date'].isna()

off_fil['cessation_before_appointment'] = off_fil['cessation_date'] < off_fil['appointment_date']
print(f"- Cessation before appointment: {off_fil['cessation_before_appointment'].sum()} filings")

off_fil = convert_to_category(off_fil, ['nationality_or_region', 'filing_status'])

display(missing_report(off_fil))
off_fil[off_fil['cessation_before_appointment']][['filing_id', 'full_name', 'appointment_date', 'cessation_date', 'filing_date']]

- Whitespace fixed in full_name: 1 value(s), 105 -> 104 distinct
- Duplicate rows removed: 1
- filing_id unique: yes
- Dates converted: appointment_date, cessation_date, filing_date
- Cessation before appointment: 9 filings


,missing,%
cessation_date,77,57.0


,filing_id,full_name,appointment_date,cessation_date,filing_date
35,FLG-000036,Calen Uren,2007-01-01,2006-12-31,2007-03-08
49,FLG-000050,Liora Corda,2014-01-01,2012-12-31,2014-02-23
76,FLG-000077,Jalen Quorin,2013-01-01,2011-12-31,2013-01-26
83,FLG-000084,Orin Norel,1998-01-01,1997-12-31,1998-01-26
99,FLG-000100,Pavel Waren,2011-01-01,2010-12-31,2011-02-03
123,FLG-000124,Elian Cevan,2025-01-01,2024-12-31,2025-03-04
126,FLG-000127,Daria Garen,2000-01-01,1998-12-31,2000-03-15
128,FLG-000129,Uma Wessen,2015-01-01,2014-12-31,2015-02-06
130,FLG-000131,Petra Koral,2005-01-01,2004-12-31,2005-03-01


**Findings**

- 1 duplicate removed (`FLG-000053`).
- Whitespace: `'Pavel  Waren '` counted as a second person (105 → 104 names).
- Birth: exact year or decade (`1920–1929`). We keep both bounds instead of inventing a precise year.
- Empty `cessation_date` = mandate still running, hence the `is_active` column.
- **9 mandates end before they start.** We do not correct them (error or backdating?): we flag them with `cessation_before_appointment`.

### `public_appointments`

In [12]:
pub_app = tables["public_appointments"]
print(pub_app.shape)
pub_app.head()

(62, 12)


,appointment_record_id,appointing_jurisdiction,office_holder_name,recorded_birth_year,office_title,institution_name,appointment_start,appointment_end,appointment_basis,departure_reason,official_region,source_notice_reference
0,APT-000001,Aveloran,Ilyan Rho,1975,Regional pollinator and agricultural...,Avelora Agricultural Resilience Agency,2003-01-01,NaN,appointed,NaN,Aveloran,NOTICE-AVL-2003-0002
1,APT-000002,Kintaran,Amara Senn,1980,Cave-access programme manager,Kintara Cave and Pollinator Authority,2005-01-01,NaN,appointed,NaN,Kintaran,NOTICE-KNT-2005-0003
2,APT-000003,Kintaran,Leora Kassen,1947,Head of government of Kintara Republic,Office of the Kintara Chancellor,1978-01-01,NaN,appointed,NaN,Kintaran,NOTICE-KNT-1978-0004
3,APT-000004,Aveloran,Tavian Serat,1953,Senior minister of the Avelora Feder...,Avelora Ministry of Strategic Resources,1984-01-01,NaN,appointed,NaN,Aveloran,NOTICE-AVL-1984-0005
4,APT-000005,Tembaran,Mirek Talven,1942,Regional governor in the Tembara Com...,Tembara Coastal Administration,01/01/1974,NaN,appointed,NaN,Tembaran,NOTICE-TMB-1974-0006


In [13]:
pub_app = clean_table(pub_app, 'appointment_record_id', date_cols=['appointment_start', 'appointment_end'])

pub_app['recorded_birth_year'] = pd.to_numeric(pub_app['recorded_birth_year'], errors='coerce').astype('Int64')

pub_app['is_active'] = pub_app['appointment_end'].isna()
print(f"- End before start: {(pub_app['appointment_end'] < pub_app['appointment_start']).sum()}")

print(f"- official_region identical to appointing_jurisdiction: {(pub_app['official_region'] == pub_app['appointing_jurisdiction']).all()}")
print(f"- appointment_basis values: {', '.join(pub_app['appointment_basis'].unique())}")

pub_app = convert_to_category(pub_app, ['appointing_jurisdiction', 'institution_name', 'appointment_basis', 'departure_reason', 'official_region'])

display(missing_report(pub_app))
pub_app[pub_app['recorded_birth_year'].isna()][['appointment_record_id', 'office_holder_name', 'office_title']]

- Whitespace fixed in institution_name: 1 value(s), 9 -> 8 distinct
- Duplicate rows removed: 0
- appointment_record_id unique: yes
- Dates converted: appointment_start, appointment_end
- End before start: 0
- official_region identical to appointing_jurisdiction: True
- appointment_basis values: appointed


,missing,%
recorded_birth_year,9,14.5
appointment_end,34,54.8
departure_reason,34,54.8


,appointment_record_id,office_holder_name,office_title
5,APT-000006,Elira Vaun,Retired political intermediary and f...
12,APT-000013,Mirek Talven,Deputy regional governor
19,APT-000020,Nora Yorin,Programme manager
26,APT-000027,Faris Quell,Regional adviser
33,APT-000034,Bela Wessen,Accreditation officer
40,APT-000041,Marek Kerin,Wildlife programme officer
47,APT-000048,Kellan Orell,Programme manager
54,APT-000055,Farah Ferran,Biodiversity programme officer
61,APT-000062,Petra Koral,Regional administrative director


**Findings**

- Whitespace: one institution counted twice, 9 → 8.
- Empty `appointment_end` = still in office, hence the `is_active` column. No end date is before its start date.
- `official_region` is a copy of `appointing_jurisdiction`, and `appointment_basis` is always `appointed`: these columns add nothing.
- `recorded_birth_year`: 9 missing, exactly one row in 7. Not imputed.

### `regulatory_case_events`

In [14]:
reg_evt = tables["regulatory_case_events"]
print(reg_evt.shape)
reg_evt.head()

(233, 16)


,regulatory_event_id,case_number,case_category,event_time,activity,subject_name,subject_registration_number,agency_name,initiating_source,assigned_unit,related_site_code,related_shipment_number,previous_status,new_status,outcome,public_reference
0,RGE-000001,REG-2018-0001,environmental_complaint,2018-05-14T09:00:00,case_opened,Avelora Agricultural Resilience Agency,NaN,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,new,open,case registered,NaN
1,RGE-000002,REG-2018-0001,environmental_complaint,2018-05-18T09:00:00,evidence_received,Avelora Agricultural Resilience Agency,NaN,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,open,evidence_review,material received from Lantern Inves...,NaN
2,RGE-000003,REG-2018-0001,environmental_complaint,2018-05-20T09:00:00,triage_review,Avelora Agricultural Resilience Agency,NaN,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,evidence_review,assigned,case assigned,NaN
3,RGE-000004,REG-2018-0001,environmental_complaint,2018-05-23T09:00:00,technical_review,Avelora Agricultural Resilience Agency,NaN,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,assigned,findings_review,technical review completed,NaN
4,RGE-000005,REG-2018-0001,environmental_complaint,2018-05-27T09:00:00,finding_issued,Avelora Agricultural Resilience Agency,NaN,Tembara Customs Coordination Unit,civil society referral,integrity review,AVL-MAR,SHP-2018-00032,findings_review,monitoring,monitoring recommendation issued,NaN


In [15]:
reg_evt = clean_table(reg_evt, 'regulatory_event_id', date_cols=['event_time'])
print(f"- Period: {reg_evt['event_time'].min():%Y-%m-%d} -> {reg_evt['event_time'].max():%Y-%m-%d}")

before = reg_evt['activity'].nunique()
reg_evt['activity'] = reg_evt['activity'].str.lower()
print(f"- activity: {before} -> {reg_evt['activity'].nunique()} distinct values")

reg_evt = convert_to_category(reg_evt, ['case_category', 'activity', 'agency_name', 'initiating_source', 'assigned_unit', 'previous_status', 'new_status', 'outcome'])

unmatched_subject = check_foreign_key(reg_evt['subject_registration_number'], reg_ent['registration_number'], 'subject_registration_number')

display(pd.crosstab(reg_evt['initiating_source'], reg_evt['assigned_unit']))
display(missing_report(reg_evt))

- Duplicate rows removed: 1
- regulatory_event_id unique: yes
- Dates converted: event_time
- Period: 2016-04-10 -> 2025-05-30
- activity: 11 -> 9 distinct values
- subject_registration_number: 159 filled, 87 not found (55%), 73 empty


assigned_unit,case review,integrity review
initiating_source,,
civil society referral,0,62
routine referral,170,0


,missing,%
subject_registration_number,73,31.5
related_site_code,110,47.4
related_shipment_number,140,60.3
public_reference,155,66.8


**Findings**

- 1 duplicate removed (`RGE-000176`).
- `activity`: `Technical_Review` → `technical_review` (11 → 9 values). Only this column is lowercased, so proper names are not damaged.
- `assigned_unit` can be deduced from `initiating_source`: only one of the two is needed.
- `subject_registration_number`: empty for the 5 public bodies involved (normal, they are not companies). **87 filled numbers (55%) cannot be found** in `registered_entities`.

### `site_access_case_events`

In [16]:
acc_evt = tables["site_access_case_events"]
print(acc_evt.shape)
acc_evt.head()

(413, 15)


,access_event_id,access_case_number,site_code,event_time,activity,requesting_organization_name,requesting_registration_number,authorizing_agency,authorized_person_name,access_purpose,equipment_category,previous_status,new_status,confidentiality_class,outcome
0,ACE-000001,ACC-2018-0001,TMB-NAC,2018-01-15T09:00:00,request_submitted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Bela Ferran,acoustic monitoring and equipment ma...,portable monitoring and cold-storage...,draft,submitted,restricted,request logged
1,ACE-000002,ACC-2018-0001,TMB-NAC,2018-01-21T09:00:00,technical_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Kellan Orell,acoustic monitoring and equipment ma...,portable monitoring and cold-storage...,submitted,technical_review,restricted,technical review completed
2,ACE-000003,ACC-2018-0001,TMB-NAC,2018-01-26T09:00:00,wildlife_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Quorin,acoustic monitoring and equipment ma...,portable monitoring and cold-storage...,technical_review,wildlife_review,restricted,wildlife conditions recorded
3,ACE-000004,ACC-2018-0001,TMB-NAC,2018-01-30T09:00:00,security_review,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Jalen Zorin,acoustic monitoring and equipment ma...,portable monitoring and cold-storage...,wildlife_review,security_review,restricted,site-security conditions recorded
4,ACE-000005,ACC-2018-0001,TMB-NAC,2018-02-05T09:00:00,authorization_granted,Horizon Field Research Cooperative,REG-AVL-01013,Tembara Biodiversity Service,Xara Lask,acoustic monitoring and equipment ma...,portable monitoring and cold-storage...,security_review,authorized,restricted,authorization issued


In [17]:
acc_evt = clean_table(acc_evt, 'access_event_id', date_cols=['event_time'])
print(f"- Period: {acc_evt['event_time'].min():%Y-%m-%d} -> {acc_evt['event_time'].max():%Y-%m-%d}")

before = acc_evt['activity'].nunique()
acc_evt['activity'] = acc_evt['activity'].str.lower()
print(f"- activity: {before} -> {acc_evt['activity'].nunique()} distinct values")

acc_evt = convert_to_category(acc_evt, ['site_code', 'activity', 'authorizing_agency', 'access_purpose', 'equipment_category', 'previous_status', 'new_status', 'confidentiality_class', 'outcome'])

unmatched_requester = check_foreign_key(acc_evt['requesting_registration_number'], reg_ent['registration_number'], 'requesting_registration_number')

display(missing_report(acc_evt))

- Whitespace fixed in access_purpose: 1 value(s), 9 -> 8 distinct


- Duplicate rows removed: 1
- access_event_id unique: yes
- Dates converted: event_time
- Period: 2015-02-15 -> 2025-06-15
- activity: 11 -> 9 distinct values
- requesting_registration_number: 347 filled, 203 not found (59%), 65 empty


,missing,%
requesting_registration_number,65,15.8
authorized_person_name,2,0.5


**Findings**

- 1 duplicate removed (`ACE-000168`).
- Whitespace: one access purpose counted twice, 9 → 8.
- `activity`: `REQUEST_SUBMITTED` → `request_submitted` (11 → 9 values). Only this column is lowercased.
- **203 filled organisation numbers (59%) cannot be found** in `registered_entities`.

### `institutional_memos`

In [18]:
memos = tables["institutional_memos"]
print(memos.shape)
memos.head()

(30, 12)


,document_id,document_date,document_type,author_name,author_organization,recipient_text,subject_line,text,related_purchase_order,related_shipment_number,related_case_number,visibility_level
0,MEM-510062,2024-04-30,inventory_explanation,Soren Kade,Meridian BioLogistics,Restricted operations and compliance...,April Biological-Unit Reconciliation,SUBJECT\nUnresolved biological-unit ...,NaN,NaN,NaN,restricted
1,MEM-510100,2023-08-17,security_instruction,Soren Kade,Meridian BioLogistics,Restricted operations and compliance...,Secure Delivery Record-Handling Inst...,SCOPE\nThis instruction applies to s...,NaN,SHP-2022-00011,NaN,restricted
2,MEM-510101,2025-03-14,meeting_record,Soren Kade,Meridian BioLogistics,Restricted operations and compliance...,Restricted CHAMELEO Governance Review,MEETING RECORD — RESTRICTED GOVERNAN...,NaN,SHP-2023-00035,NaN,restricted
3,MEM-510128,2018-05-15,internal_memorandum,Adrian Vosmere,Vosmere Biotechnics,"Operations, finance, and compliance",Internal Operations Memorandum 56,SUBJECT\nOperational review of cold-...,NaN,SHP-2020-00019,NaN,restricted
4,MEM-510069,2021-10-22,internal_memorandum,Dr. Elias Voss,Meridian BioLogistics,"Operations, finance, and compliance",Internal Operations Memorandum 57,SUBJECT\nOperational review of suppl...,NaN,NaN,NaN,internal


In [19]:
memos = clean_table(memos, 'document_id', date_cols=['document_date'], trim_exclude=['text'])
memos = convert_to_category(memos, ['document_type', 'author_organization', 'recipient_text', 'visibility_level'])

display(missing_report(memos))

- Whitespace fixed in subject_line: 1 value(s), 30 -> 30 distinct
- Duplicate rows removed: 0
- document_id unique: yes
- Dates converted: document_date


,missing,%
related_purchase_order,25,83.3
related_shipment_number,22,73.3
related_case_number,20,66.7


**Findings**

- 30 documents, no duplicates.
- `text` is not cleaned: its line breaks give the document its structure.
- Whitespace fixed in one `subject_line`.
- `related_purchase_order` and `related_shipment_number` point to the `logistics` bundle (not checked here).

### `public_and_investigative_documents`

In [20]:
pub_docs = tables["public_and_investigative_documents"]
print(pub_docs.shape)
pub_docs.head()

(40, 11)


,document_id,document_date,collection,document_type,author_name,author_organization,named_location,title,text,public_or_restricted,source_reference
0,PUB-510045,2022-07-08,investigative_repository,confidential_briefing,Dr. Lysa Quen,Orison Medical Audit Board,Valenne Central District,Restricted Review of Recurrent Execu...,PURPOSE\nThis restricted briefing su...,restricted,ARC-2022-0103
1,PUB-510094,2024-07-12,investigative_repository,research_summary,Selene Aras,Threshold Earth Observatory,Threshold Earth Avelora Laboratory,Opposed Ecological Strategies and Di...,QUESTION\nWhy do two groups using si...,public,ARC-2024-0107
2,PUB-510043,1942-11-06,public_archive,historical_letter,Emil Warden,Warden Medical Ventures,Valenne Republic,Letter from the Vosmere Commercial P...,"6 November 1942\n\nDear M.,\n\nYour ...",public,ARC-1942-0108
3,PUB-510033,1962-05-18,public_archive,meeting_extract,Jonah Bell,Bell Conservation Survey,Avelora Federation,Extract from the Eastern Orchard Pro...,EXTRACT FROM MINUTES — 18 MAY 1962\n...,public,ARC-1962-0109
4,PUB-510044,2025-02-21,investigative_repository,investigative_note,Jora Menne,Lantern Investigative Bureau,Valenne Central District,Jacob Vale Identity-Continuity Lead,LEAD\nJacob Vale may be connected to...,restricted,ARC-2025-0111


In [21]:
pub_docs = clean_table(pub_docs, 'document_id', date_cols=['document_date'], trim_exclude=['text'])

pub_docs['collection_grouped'] = pub_docs['collection'].replace({
    'investigative_archive': 'investigative_repository',
})

pub_docs = convert_to_category(pub_docs, ['collection', 'collection_grouped', 'document_type', 'author_organization', 'named_location', 'public_or_restricted'])

display(missing_report(pub_docs))

- Duplicate rows removed: 0
- document_id unique: yes
- Dates converted: document_date


,missing,%
author_name,1,2.5


**Findings**

- 40 documents, no duplicates.
- `text` is not cleaned: its line breaks give the document its structure.
- `investigative_archive` (1 document) is merged with `investigative_repository` in `collection_grouped`: same `ARC` reference series. This is an analysis choice, not a correction. The original `collection` column is kept.
- `public_archive` is not merged with `historical_archive`: its documents have `ARC` references, like the investigation files.

## Consistency between tables

Countries are written in two ways: `Valenne Republic` or `Valennan`. The `region_code` column gives them a single code (`VLN`) so all tables can be grouped by country.

In [22]:
REGION_CODES = {
    'Valenne Republic': 'VLN', 'Valennan': 'VLN',
    'Avelora Federation': 'AVL', 'Aveloran': 'AVL',
    'Tembara Commonwealth': 'TMB', 'Tembaran': 'TMB',
    'Nordhaven Union': 'NDH', 'Nordhaven': 'NDH',
    'Kintara Republic': 'KNT', 'Kintaran': 'KNT',
}

reg_ent['region_code'] = reg_ent['jurisdiction'].astype(str).map(REGION_CODES)
lic_fac['region_code'] = lic_fac['region'].astype(str).map(REGION_CODES)
lic_pro['region_code'] = lic_pro['registered_region'].astype(str).map(REGION_CODES)
off_fil['region_code'] = off_fil['nationality_or_region'].astype(str).map(REGION_CODES)
pub_app['region_code'] = pub_app['official_region'].astype(str).map(REGION_CODES)

Share of company numbers that cannot be found in `registered_entities`:

In [23]:
fk = pd.DataFrame({
    'filled': [off_fil['registration_number'].notna().sum(),
               reg_evt['subject_registration_number'].notna().sum(),
               acc_evt['requesting_registration_number'].notna().sum()],
    'not_found': [(~off_fil['registration_number'].isin(reg_ent['registration_number'])).sum(),
                  unmatched_subject.sum(),
                  unmatched_requester.sum()],
}, index=['officer_filings', 'regulatory_case_events', 'site_access_case_events'])
fk['%'] = (fk['not_found'] / fk['filled'] * 100).round().astype(int)
fk

,filled,not_found,%
officer_filings,135,0,0
regulatory_case_events,159,87,55
site_access_case_events,347,203,59


**Finding**: 0% for `officer_filings`, but 55% and 59% for the two event logs. Many organisations involved in cases are not in the register: a lead for the analysis.

## Analysis: part of the secret

We start from what we noticed during cleaning, and pull the thread.

### 1. Licences that are too old

When converting the dates of `licensed_professionals`, we saw some very old licences. We sort by issue date:

In [24]:
lic_pro.sort_values('license_issue_date').head(3)[['professional_name', 'license_issue_date', 'license_status']]

,professional_name,license_issue_date,license_status
6,J. Varlan,1886-07-01,active
0,Emil Warden,1895-07-01,active
7,Jonah Bell,1924-07-01,active


Two licences date from the 19th century and are **still active**. We look for these two people among the officers:

In [25]:
off_fil[off_fil['full_name'].isin(['Emil Warden', 'J. Varlan'])][['full_name', 'birth_year_or_age_band', 'appointment_date', 'cessation_date']]

,full_name,birth_year_or_age_band,appointment_date,cessation_date
1,Emil Warden,1871,1895-01-01,1943-12-31
4,J. Varlan,1862,1885-01-01,1918-12-31


Emil Warden was born in 1871 and J. Varlan in 1862: they would be over 150 years old today. Did Emil Warden write any documents?

In [26]:
pub_docs[pub_docs['author_name'] == 'Emil Warden'][['document_id', 'document_date', 'title']]

,document_id,document_date,title
2,PUB-510043,1942-11-06,Letter from the Vosmere Commercial P...
39,PUB-510046,1914-07-04,Private Historical Note 114


In [27]:
print(pub_docs.loc[pub_docs['document_id'] == 'PUB-510043', 'text'].iloc[0])

6 November 1942

Dear M.,

Your note reached me at the old Wren Street address, though the forwarding clerk again placed it among the natural-history circulars. Please use that address only for matters concerning the southern holdings. Mr. V. has agreed that the associate from the coastal enterprise may attend the winter meeting, provided the minutes describe him as a commercial adviser.

The present shortages have encouraged several directors to abandon the long programme. I have advised patience. The advantage is not in a single shipment but in arranging the conditions under which future supply becomes ordinary. We used the same phrase in Valenne: make the route commonplace, and the exceptional passage will cease to attract notice.

I shall travel under the Vosmere papers. The medical enterprise has retained our earlier correspondence, so do not mention the prior name in a reply. Send the revised investment schedule to the Kestrel office and ask whether J. still objects to the expans

In this 1942 letter, he writes: *"I shall travel under the **Vosmere** papers… do not mention the prior name"*. So he changes his name. The archival note adds that the same clues come back with **Elias Voss**. We look for these names among the officers:

In [28]:
off_fil[off_fil['full_name'].str.contains('Warden|Vosmere|Voss')][['full_name', 'birth_year_or_age_band', 'appointment_date', 'cessation_date']].sort_values('appointment_date')

,full_name,birth_year_or_age_band,appointment_date,cessation_date
1,Emil Warden,1871,1895-01-01,1943-12-31
2,Adrian Vosmere,1920–1929,1952-01-01,1991-12-31
3,Dr. Elias Voss,1961,1996-01-01,NaT


The mandates follow one another: Warden until 1943, Vosmere from 1952 to 1991, Voss since 1996. Another document mentions Vosmere:

In [29]:
pub_docs[pub_docs['text'].str.contains('Vosmere')][['document_id', 'document_date', 'title']]

,document_id,document_date,title
2,PUB-510043,1942-11-06,Letter from the Vosmere Commercial P...
7,PUB-510076,2025-02-05,Catalog Note on Recurrent Historical...


In [30]:
print(pub_docs.loc[pub_docs['document_id'] == 'PUB-510076', 'text'].iloc[0])

PROVENANCE
This catalog note covers selected commercial, medical-enterprise, and natural-history papers dated 1911–1971. The originals come from three separately acquired collections and have not been merged physically.

CATALOG FINDING
Records attributed to Adrian Vosmere, Jonah Bell, and Julien Marot repeatedly use the Wren Street forwarding address, refer to the same two commercial associates, and employ the phrase “make the conditions ordinary” in discussions of routes and ecological infrastructure. Signature comparison is suggestive but not conclusive because several items are copies or secretary-written drafts. Travel intervals overlap less than expected for three independent advisers, yet the archive contains gaps large enough to permit ordinary explanations.

INTERPRETATION
The bundle justifies an alias-continuity hypothesis and provides a bridge for later identity-resolution work. It does not certify that the names represent one person, and it does not connect the historical r

Investigators noticed that Vosmere, Bell and Marot use the same address and the same phrases: they suspect **a single person using several names**.

What remains is to understand **how** this person does not age. The 1942 letter gives a clue: Warden talks about a *supply* to be hidden in ordinary routes, *"make the route commonplace, and the exceptional passage will cease to attract notice"*. If something is hidden in everyday activities, we should look for what stands out in the activity logs.

### 2. Visits at 3 a.m.

`site_access_case_events` records visits to natural sites: very ordinary activities, at first sight. Since `event_time` is now a real date, we can look at what time they happen:

In [31]:
acc_evt['event_time'].dt.hour.value_counts().sort_index()

event_time
3      18
9     286
11     54
17     36
18     18
Name: count, dtype: int64

18 events happen at **3 a.m.** Who is involved?

In [32]:
night = acc_evt[acc_evt['event_time'].dt.hour == 3]
print(night['requesting_organization_name'].value_counts())
night[['access_case_number', 'site_code', 'authorizing_agency', 'equipment_category']].head()

requesting_organization_name
Horizon Field Research Cooperative    18
Name: count, dtype: int64


,access_case_number,site_code,authorizing_agency,equipment_category
6,ACC-2018-0001,TMB-NAC,Tembara Biodiversity Service,portable monitoring and cold-storage...
14,ACC-2018-0002,KNT-KES,Kintara Cave and Pollinator Authority,portable monitoring and cold-storage...
22,ACC-2018-0003,AVL-MAR,Avelora Agricultural Resilience Agency,protective field equipment
29,ACC-2019-0004,TMB-PEL,Tembara Biodiversity Service,portable monitoring and cold-storage...
36,ACC-2019-0005,KNT-LUM,Kintara Cave and Pollinator Authority,protective field equipment


It is always **Horizon Field Research Cooperative**, often with **cold-storage equipment**. To understand why, we look at the whistleblower messages, one of the document types:

In [33]:
pub_docs[pub_docs['document_type'] == 'whistleblower_message'][['document_id', 'document_date', 'author_name', 'title']]

,document_id,document_date,author_name,title
5,PUB-510079,2025-04-09,Mara Venn,Message Concerning Collected Eonycteris
30,PUB-510023,2021-10-07,Miran Zorin,Source Message 105


In [34]:
print(pub_docs.loc[pub_docs['document_id'] == 'PUB-510079', 'text'].iloc[0])

I worked on restricted receiving and reconciliation records. I am sending this because the standard files make the programme look like conservation support, but the handling instructions say something different.

The consignments include live or recently collected *Eonycteris spelaea*. They arrive under wildlife-monitoring or biological-sample descriptions, are consolidated in sealed units, and then disappear from the ordinary inventory sequence. I saw preparation sheets that used the word consumption and matched them to access cases, cold-storage receipts, and adjustment numbers. The animals were not being transferred only for tagging, veterinary examination, or habitat research.

Useful references include the spring Kestrel access file, the secure delivery series, and the biological-unit adjustments reviewed by the warehouse controller. Some copies use CHAMELEO routing, but I cannot explain who controls the entire programme. I did not see a complete destination record, and I cannot c

Bats (*Eonycteris spelaea*) are collected under the cover of wildlife monitoring, kept cold, then disappear from the inventory. The sheets mention **consumption**. The message cites a code: **CHAMELEO**.

### 3. The CHAMELEO code

We search for this word in the memos:

In [35]:
memos[memos['text'].str.contains('CHAMELEO')][['document_id', 'document_date', 'author_organization', 'subject_line']]

,document_id,document_date,author_organization,subject_line
1,MEM-510100,2023-08-17,Meridian BioLogistics,Secure Delivery Record-Handling Inst...
2,MEM-510101,2025-03-14,Meridian BioLogistics,Restricted CHAMELEO Governance Review
22,MEM-510089,2019-04-13,Nivren Maritime Logistics,Records Security Instruction 75


In [36]:
print(memos.loc[memos['document_id'] == 'MEM-510101', 'text'].iloc[0])

MEETING RECORD — RESTRICTED GOVERNANCE REVIEW

Attendees included the executive chair, the secure-delivery manager, clinical programme counsel, and the records secretary.

AGENDA
1. Authority for CHAMELEO routing.
2. Relationship between programme designation and executive approvals.
3. Separation of ordinary corporate records from restricted appendices.

DISCUSSION
The chair confirmed that Unwabu is the designation used in the restricted governance schedule for the authority approving CHAMELEO exceptions. The same schedule assigns final escalation to the Meridian executive office. Participants agreed that ordinary employees should see only shipment, inventory, and compliance fields needed for their roles. The secretary noted that earlier instructions treated CHAMELEO as a handling code, while the governance schedule links it to a named authority.

DECISIONS
The secure-delivery manager will maintain the restricted appendix. Compliance will retain a cross-reference between the designati

CHAMELEO is a confidential delivery route, controlled by the **executive office of Meridian BioLogistics**. Who runs Meridian?

In [37]:
off_fil[off_fil['registration_number'] == 'REG-VLN-01001'][['full_name', 'role_title', 'appointment_date', 'cessation_date']]

,full_name,role_title,appointment_date,cessation_date
3,Dr. Elias Voss,Executive chair of Meridian BioLogis...,1996-01-01,NaT
9,Soren Kade,Strategic logistics director at Meri...,1999-01-01,NaT
28,Nora Essen,Operations director,1997-01-01,2016-12-31
55,Alina Istra,Compliance analyst,2010-01-01,NaT
82,Orin Norel,Cold-chain programme manager,1998-01-01,NaT
109,Vera Faron,Operations director,1990-01-01,NaT


The executive chair is **Dr. Elias Voss**, the last name in lead 1. The two leads meet.

### Takeaways

- One person seems to go through the centuries by changing names: Warden → Vosmere → … → **Voss**.
- Bats are collected at night to be "consumed", through a route run by Voss's company.
- **Hypothesis**: these bats would be used to stay young.

These are only clues (names, dates, documents), not proof. The other bundles (`ecology`, `logistics`, `health`) will help check them.